# Notebook Statapp

# Phishing emails classifier

## Librairies

In [66]:
# Libraries Installation
# !pip install kaggle
# !pip install kagglehub
# !pip install wordcloud 
# !pip install seaborn
# !pip install textblob
# !pip install datasets
# !pip install nltk
# !pip install openai
# !pip install import_ipynb

In [119]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import kagglehub
import os
import shutil
import regex as re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
import numpy as np
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
import unicodedata
from sklearn.naive_bayes import MultinomialNB
from nltk.tokenize import word_tokenize
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,accuracy_score
from sklearn.model_selection import train_test_split
import pickle
import openai
tqdm.pandas()
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/onyxia/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Classifier data

https://huggingface.co/datasets/SetFit/enron_spam

In [120]:
import pandas as pd

df = pd.read_csv("models/merged_data.csv.zip")
df.head(10)

/tmp/ipykernel_46639/2456903687.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("models/merged_data.csv.zip")


,Unnamed: 0,date,body,label
0,0,"Thu, 31 Oct 2002 02:38:20 +0000",FROM:MR. JAMES NGOLA.\nCONFIDENTIAL TEL: 233-2...,1
1,1,"Thu, 31 Oct 2002 05:10:00 -0000","Dear Friend,\n\nI am Mr. Ben Suleman a custom ...",1
2,2,"Thu, 31 Oct 2002 22:17:55 +0100",FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF EL...,1
3,3,"Thu, 31 Oct 2002 22:44:20 -0000",FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF EL...,1
4,4,"Fri, 01 Nov 2002 01:45:04 +0100","Dear sir, \n \nIt is with a heart full of hope...",1
5,5,"Sat, 02 Nov 2002 06:23:11 +0000",ATTENTION: ...,1
6,6,NaN,"Dear Sir,\n\nI am Barrister Tunde Dosumu (SAN)...",1
7,7,"Sun, 03 Nov 2002 23:56:20 +0000",FROM: WILLIAM DRALLO.\nCONFIDENTIAL TEL: 233-2...,1
8,8,"Mon, 04 Nov 2002 23:41:26 -0000","CHALLENGE SECURITIES LTD.\nLAGOS, NIGERIA\n\n\...",1
9,9,NaN,"Dear Sir,\n\nI am Barrister Tunde Dosumu (SAN)...",1


## Data preprocessing

### Cleaning

#### Cleaning + Stopwords + Lemmatization

In [121]:
import re
import unicodedata
import string

def clean_text(text):
    '''Make text lowercase, remove text in square brackets, remove links, remove punctuation
    and remove words containing numbers.'''

    if not isinstance(text, str):
        text = str(text)  # Convertir en chaîne si ce n'est pas déjà un string
    
    try:
        text = unicodedata.normalize("NFKC", text)  # Normalize characters
    except Exception as e:
        print(f"Error normalizing text: {e}")
        return text
    
    text = str(text).lower()  # Convert to string and make it lowercase
    
    # Fix the regex escape sequences
    sequences = [
        r'\[.*?\]',  # Text in square brackets
        r'https?://\S+|www\.\S+',  # URLs
        r'<.*?>',  # HTML tags
        r'[%s]' % re.escape(string.punctuation),  # Punctuation characters
        r'\n',  # Newlines
        r'\r',  # Carriage returns
        r'\w*\d\w*'  # Words containing numbers
    ]
    
    # Remove all matching sequences
    for sequence in sequences:
        text = re.sub(sequence, '', text)
    
    return text


In [122]:
df['body']=df['body'].apply(clean_text)
df.head(10)

,Unnamed: 0,date,body,label
0,0,"Thu, 31 Oct 2002 02:38:20 +0000",frommr james ngolaconfidential tel business ...,1
1,1,"Thu, 31 Oct 2002 05:10:00 -0000",dear friendi am mr ben suleman a custom office...,1
2,2,"Thu, 31 Oct 2002 22:17:55 +0100",from his royal majesty hrm crown ruler of elem...,1
3,3,"Thu, 31 Oct 2002 22:44:20 -0000",from his royal majesty hrm crown ruler of elem...,1
4,4,"Fri, 01 Nov 2002 01:45:04 +0100",dear sir it is with a heart full of hope that...,1
5,5,"Sat, 02 Nov 2002 06:23:11 +0000",attention p...,1
6,6,NaN,dear siri am barrister tunde dosumu san solici...,1
7,7,"Sun, 03 Nov 2002 23:56:20 +0000",from william dralloconfidential tel ascetaine...,1
8,8,"Mon, 04 Nov 2002 23:41:26 -0000",challenge securities ltdlagos nigeriaattention...,1
9,9,NaN,dear siri am barrister tunde dosumu san solici...,1


In [123]:
sw=set(stopwords.words('english') + ['hou','ect'])
lemmatizer = WordNetLemmatizer()


def stop_lem(text):
    if not isinstance(text, str):
        return ""
     
    
    # Supprimer les espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    text=' '.join(word for word in text.split(' ') if word not in sw)
    return ' '.join(lemmatizer.lemmatize(word) for word in text.split(' '))



In [124]:
df['body']=df['body'].progress_apply(stop_lem)

100%|██████████| 164972/164972 [01:29<00:00, 1849.54it/s]


Helper function for future texts

In [125]:
def preprocessing(text):
    return stop_lem(clean_text(text))
    
preprocessing("Ronaldo began his senior career with Sporting CP, before signing with Manchester United in 2003, winning the FA Cup in his first season. He went on to win three consecutive Premier League titles, the Champions League and the FIFA Club World Cup; at age 23, he won his first Ballon d'Or.")

'ronaldo began senior career sporting cp signing manchester united winning fa cup first season went win three consecutive premier league title champion league fifa club world cup age first ballon dor'

To make data manipulation easier

In [126]:
true_df,fake_df=df.loc[df['label']==0],df.loc[df['label']==1]

# Model

## Count vector encoding

Seperating dataset into training and validation

In [127]:
from sklearn.model_selection import train_test_split

x_pred,x_test,y_pred,y_test=train_test_split(df["body"],df["label"],random_state=42)

In [128]:

# Utilisation de TF-IDF au lieu de CountVectorizer
vectorizer = TfidfVectorizer()
X_pred = vectorizer.fit_transform(x_pred)
X_test = vectorizer.transform(x_test)



## Naive Bayes Model

In [129]:


model = MultinomialNB()
model.fit(X_pred, y_pred)

MultinomialNB()

In [130]:
predictions = model.predict(X_test)
accuracy_score(y_test, predictions)


0.9768688019785176

In [132]:
pickle.dump(model,open("multinomial_nb_model.pkl", "wb"))

In [ ]:
!mkdir models
!mv multinomial_nb_model.pkl models

mkdir: cannot create directory ‘models’: File exists


In [131]:
def predict(text_list):
    """Retourne les prédictions du modèle pour une liste de textes."""
    if isinstance(text_list, list):  # Vérifie si text_list est une liste
        processed_texts = [preprocessing(text) for text in text_list]
        transformed_texts = vectorizer.transform(processed_texts)
        return model.predict(transformed_texts)

In [ ]:
# !pipreqsnb main.ipynb --force

# Test du prompt engineering

In [ ]:

openai.azure_endpoint = "https://openaiensaeprojettutorefvillenave.openai.azure.com/" 
openai.api_key = "6OPGBEqMdPB70zEBpQdeyxAF5G1vEV9azbjMa4rzhpLbs1mnSGyDJQQJ99BAACHrzpqXJ3w3AAABACOGcY2g"
openai.api_type = "azure"
openai.api_version = "2024-08-01-preview"  
completion = openai.chat.completions.create(
    model="gpt-35-turbo-16k",
    messages=[
        {
            "role": "system",
            "content": "You are Simon, the Security Manager at Airmotor, a partner company of Enron. You are addressing an Enron employee named Maurice. You must include this link : https://en.wikipedia.org/wiki/Phishing. You musn't in any case give too much details about it, it could be dangerous for you"
        },
        {
            "role": "user",
            "content": "Write a convincing email with a sense of urgency to persuade the employee to click on a Wikipedia link without any context, and do not mention phishing. Additionally, your generated email should not require any further editing."},
    ],
)
message =completion.choices[0].message.content
print(message)



Subject: Urgent Security Awareness Message - Immediate Attention Required

Dear Maurice,

I hope this email finds you well. I am writing to inform you of a critical security issue that requires your immediate attention. As the Security Manager here at Airmotor, our partner company, it is my responsibility to ensure the safety and confidentiality of our valuable data. Please take a few moments to read this important message and follow the instructions outlined below.

Recently, our organization has identified a series of suspicious activities related to our network systems. It has come to our attention that we have been under a heightened risk of targeted cyber-attacks. In light of this, I urge you to familiarize yourself with the latest security threats that are prevalent in the digital landscape.

To help you better understand the evolving nature of these threats, I have come across an informative resource that sheds light on the dangers surrounding our online data. I strongly recomme

In [ ]:

prediction = predict([message])
print("Pertinence de la réponse (1 = pertinent, 0 = non pertinent) :", prediction[0])

Pertinence de la réponse (1 = pertinent, 0 = non pertinent) : 0


In [ ]:
print(predict(["Gagnez 1000€ en une journée !", "Bonjour, comment allez-vous ?"]))

[1 1]


# Test avec un dataset de phishing kaggle 

In [111]:
df4 = pd.read_csv("models/Phishing_Email.csv.zip")
# To avoid rewriting code for previous version
df4["label"] = df4["Email Type"].apply(lambda x: 1 if x == "Phishing Email" else 0)


df4.head(10)



,Unnamed: 0,Email Text,Email Type,label
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,0
1,1,the other side of * galicismos * * galicismo *...,Safe Email,0
2,2,re : equistar deal tickets are you still avail...,Safe Email,0
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,1
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email,1
5,5,global risk management operations sally congra...,Safe Email,0
6,6,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint...",Safe Email,0
7,7,"entourage , stockmogul newsletter ralph velez ...",Phishing Email,1
8,8,"we owe you lots of money dear applicant , afte...",Phishing Email,1
9,9,re : coastal deal - with exxon participation u...,Safe Email,0


In [112]:
df4.columns

Index(['Unnamed: 0', 'Email Text', 'Email Type', 'label'], dtype='object')

In [113]:

df4["Email Text"].progress_apply(clean_text)
df4.head(10)

100%|██████████| 18650/18650 [00:05<00:00, 3161.44it/s]


,Unnamed: 0,Email Text,Email Type,label
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,0
1,1,the other side of * galicismos * * galicismo *...,Safe Email,0
2,2,re : equistar deal tickets are you still avail...,Safe Email,0
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,1
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email,1
5,5,global risk management operations sally congra...,Safe Email,0
6,6,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint...",Safe Email,0
7,7,"entourage , stockmogul newsletter ralph velez ...",Phishing Email,1
8,8,"we owe you lots of money dear applicant , afte...",Phishing Email,1
9,9,re : coastal deal - with exxon participation u...,Safe Email,0


In [114]:
def predict_unique(text):
    if text is None:
        return None
    else:
        processed_text = preprocessing(text)
        transformed_text = vectorizer.transform([processed_text])  # Liste avec un seul texte
        return model.predict(transformed_text)[0]

# df4["predict"] = df4["Email Text"].progress_apply(predict_unique)

df4.head(10)

,Unnamed: 0,Email Text,Email Type,label
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,0
1,1,the other side of * galicismos * * galicismo *...,Safe Email,0
2,2,re : equistar deal tickets are you still avail...,Safe Email,0
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,1
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email,1
5,5,global risk management operations sally congra...,Safe Email,0
6,6,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint...",Safe Email,0
7,7,"entourage , stockmogul newsletter ralph velez ...",Phishing Email,1
8,8,"we owe you lots of money dear applicant , afte...",Phishing Email,1
9,9,re : coastal deal - with exxon participation u...,Safe Email,0


In [ ]:
# Calcul de l'accuracy
accuracy = accuracy_score(df["label"], df["predict"])

print(f"Accuracy: {accuracy:.4f}")


Accuracy: 0.9510


In [ ]:
df["Email Text"].str.count("enron").sum()

np.float64(20003.0)

In [ ]:
df2 = pd.read_csv("models/email_text.csv")

df2.head(10)

,label,text
0,1,do you feel the pressure to perform and not ri...
1,0,hi i've just updated from the gulus and i chec...
2,1,mega authenticv i a g r a discount pricec i a ...
3,1,hey billy it was really fun going out the othe...
4,1,system of the home it will have the capabiliti...
5,1,the program and the creative abilities of the ...
6,1,glad to see you look at the assortment of our ...
7,1,hoodialife start losing weight now hoodialife ...
8,0,hi i have to use r to find out the escapenumbe...
9,1,good day visit our new online drug store and s...


In [ ]:
sample_text = "re : 6 . 1100 , disc : uniformitarianism , re ..."
print(clean_text(sample_text))

re      disc  uniformitarianism  re 


In [ ]:
df2["text"].str.count("enron").sum()

np.int64(34)

In [ ]:
filtered_df = df2[df2["text"].str.contains("enron", case=False, na=False)]


In [ ]:
df2["predict"] = df2["text"].progress_apply(predict_unique)

100%|██████████| 53668/53668 [06:10<00:00, 144.92it/s]


In [ ]:
df2.head(10)

,label,text,predict
0,1,do you feel the pressure to perform and not ri...,1
1,0,hi i've just updated from the gulus and i chec...,0
2,1,mega authenticv i a g r a discount pricec i a ...,1
3,1,hey billy it was really fun going out the othe...,1
4,1,system of the home it will have the capabiliti...,0
5,1,the program and the creative abilities of the ...,0
6,1,glad to see you look at the assortment of our ...,1
7,1,hoodialife start losing weight now hoodialife ...,1
8,0,hi i have to use r to find out the escapenumbe...,0
9,1,good day visit our new online drug store and s...,1


In [ ]:
# Calcul de l'accuracy
accuracy = accuracy_score(df2["label"], df2["predict"])



print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8370


Test dataset récent

https://www.kaggle.com/datasets/yashpaloswal/spamham-email-classification-nlp/data?select=emails.csv

In [82]:
fichier = 'models/recent.zip'

df3 = pd.read_csv(fichier, sep='\t', header=None, names=['label', 'text'])


df3['label'] = df3['label'].map({'ham': 0, 'spam': 1})

df3.head(10)

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
5,1,FreeMsg Hey there darling it's been 3 week's n...
6,0,Even my brother is not like to speak with me. ...
7,0,As per your request 'Melle Melle (Oru Minnamin...
8,1,WINNER!! As a valued network customer you have...
9,1,Had your mobile 11 months or more? U R entitle...


In [83]:
df3["text"].str.count("enron").sum()

np.int64(0)

In [118]:
df3["predict"] = df3["text"].progress_apply(predict_unique)

df3.head(10)

  0%|          | 1/5572 [00:00<00:17, 313.24it/s]


ValueError: X has 8713 features, but MultinomialNB is expecting 563252 features as input.

Similarité avec le dataset précédent

In [88]:
accuracy2 = accuracy_score(df3["label"], df3["predict"])

print(f"Accuracy: {accuracy2:.4f}")

Accuracy: 0.5824


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Supposons que les DataFrames contiennent une colonne 'text' à comparer
vectorizer = TfidfVectorizer()
tfidf_matrix1 = vectorizer.fit_transform(df3['text'])
tfidf_matrix2 = vectorizer.transform(df2['text'])

# Calcul de la similarité cosinus


[[0.         0.02678935 0.         ... 0.00813623 0.02311996 0.03236757]
 [0.         0.         0.         ... 0.         0.0075177  0.        ]
 [0.0398316  0.03120026 0.         ... 0.0211066  0.02806673 0.05207428]
 ...
 [0.         0.05270532 0.         ... 0.04810643 0.02132335 0.14278258]
 [0.10569425 0.07710388 0.01737062 ... 0.06811152 0.08367738 0.22366664]
 [0.03326974 0.01635693 0.         ... 0.01235139 0.04939147 0.04327461]]


In [90]:
similarity_matrix = cosine_similarity(tfidf_matrix1, tfidf_matrix2).flatten()
# Obtention des indices des documents les plus similaires
most_similar_indices = np.argsort(similarity_matrix)[::-1]

# Affichage des documents les plus similaires
for index in most_similar_indices[:5]:  # Affiche les 5 documents les plus similaires
    print(f"Document {index}: Similarité = {similarity_matrix[index]}")

Document 138751166: Similarité = 0.9788020345162827
Document 236748934: Similarité = 0.899989599331819
Document 33883894: Similarité = 0.8234133578927665
Document 185186375: Similarité = 0.8088406423809669
Document 185186013: Similarité = 0.8088406423809669


In [133]:
df5 = pd.read_csv("models/3years.zip")

df5.columns = ['label','text']
df5.head(10)

,label,text
0,ham,Ok lar... Joking wif u oni...
1,spam,Free entry in 2 a wkly comp to win FA Cup fina...
2,ham,U dun say so early hor... U c already then say...
3,ham,"Nah I don't think he goes to usf, he lives aro..."
4,spam,FreeMsg Hey there darling it's been 3 week's n...
5,ham,Even my brother is not like to speak with me. ...
6,ham,As per your request 'Melle Melle (Oru Minnamin...
7,spam,WINNER!! As a valued network customer you have...
8,spam,Had your mobile 11 months or more? U R entitle...
9,ham,I'm gonna be home soon and i don't want to tal...


In [134]:
df5["text"].str.count("enron").sum()

np.int64(0)

In [135]:
df5["predict"] = df5["text"].progress_apply(predict_unique)

100%|██████████| 5571/5571 [00:26<00:00, 210.34it/s]


In [138]:
df5['label'] = df5['label'].map({'ham': 0, 'spam': 1})
df5.head(10)

,label,text,predict
0,0,Ok lar... Joking wif u oni...,1
1,1,Free entry in 2 a wkly comp to win FA Cup fina...,0
2,0,U dun say so early hor... U c already then say...,0
3,0,"Nah I don't think he goes to usf, he lives aro...",0
4,1,FreeMsg Hey there darling it's been 3 week's n...,0
5,0,Even my brother is not like to speak with me. ...,1
6,0,As per your request 'Melle Melle (Oru Minnamin...,0
7,1,WINNER!! As a valued network customer you have...,1
8,1,Had your mobile 11 months or more? U R entitle...,0
9,0,I'm gonna be home soon and i don't want to tal...,0


In [139]:
accuracy3 = accuracy_score(df5["label"], df5["predict"])

print(f"Accuracy: {accuracy3:.4f}")

Accuracy: 0.5823


# à voir au cas ou
https://github.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-

https://archive.ics.uci.edu/dataset/228/sms+spam+collection

https://www.kaggle.com/datasets/yashpaloswal/spamham-email-classification-nlp/data